In [1]:
%%capture
!pip install unsloth
!pip install --force-reinstall --no-deps "trl<0.15"

In [2]:
!nvidia-smi

Mon Mar  2 18:40:47 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   32C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
from unsloth import FastLanguageModel
import torch

# Configuration
max_seq_length = 2048  # Context window - how many tokens the model sees at once
dtype = None           # Auto-detect: float16 for T4, bfloat16 for newer GPUs
load_in_4bit = True    # KEY: This is quantization - shrinks model from ~16GB to ~5GB

# Load Llama 3.2 3B (fits easily on free T4)
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-3B-Instruct",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Unsloth: Could not find `steps_per_generation` in grpo_trainer
Unsloth: Could not find `generation_batch_size` in grpo_trainer
==((====))==  Unsloth 2026.2.1: Fast Llama patching. Transformers: 4.57.6.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.35G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

In [4]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,                         # LoRA rank - higher = more capacity but more memory
    target_modules = [              # Which layers to adapt
        "q_proj", "k_proj", "v_proj", "o_proj",  # Attention layers
        "gate_proj", "up_proj", "down_proj",      # MLP layers
    ],
    lora_alpha = 16,                # Scaling factor (usually = r)
    lora_dropout = 0,               # No dropout for small datasets
    bias = "none",                  # Don't train bias terms
    use_gradient_checkpointing = "unsloth",  # Reduces VRAM by ~30%
    random_state = 3407,
)

Unsloth 2026.2.1 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


In [5]:
from datasets import load_dataset

# Using a product-relevant dataset - you could swap this for your own data
dataset = load_dataset("yahma/alpaca-cleaned", split="train")

# Format data into the chat template the model expects
alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

EOS_TOKEN = tokenizer.eos_token  # Tells the model when to stop generating

def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]
    texts = []
    for instruction, input, output in zip(instructions, inputs, outputs):
        text = alpaca_prompt.format(instruction, input, output) + EOS_TOKEN
        texts.append(text)
    return {"text": texts}

dataset = dataset.map(formatting_prompts_func, batched=True)

print(f"Training samples: {len(dataset)}")
print(f"\nExample:\n{dataset[0]['text'][:500]}")

README.md: 0.00B [00:00, ?B/s]

alpaca_data_cleaned.json:   0%|          | 0.00/44.3M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/51760 [00:00<?, ? examples/s]

Map:   0%|          | 0/51760 [00:00<?, ? examples/s]

Training samples: 51760

Example:
Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
Give three tips for staying healthy.

### Input:


### Response:
1. Eat a balanced and nutritious diet: Make sure your meals are inclusive of a variety of fruits and vegetables, lean protein, whole grains, and healthy fats. This helps to provide your body with the essential nutrients to function at its best and can help pr


In [7]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,              # True = pack multiple examples per sequence (faster)
    args = TrainingArguments(
        per_device_train_batch_size = 2,     # Samples per GPU per step
        gradient_accumulation_steps = 4,     # Effective batch size = 2 * 4 = 8
        warmup_steps = 5,                    # Learning rate warmup
        max_steps = 60,                      # Keep short for demo (full run: num_train_epochs=1)
        learning_rate = 2e-4,                # Standard for LoRA fine-tuning
        fp16 = not is_bfloat16_supported(),  # Mixed precision - uses T4's Tensor Cores
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,                   # Log loss every step
        optim = "adamw_8bit",                # 8-bit optimizer saves ~2GB VRAM
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

# Check memory before training
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU: {gpu_stats.name}")
print(f"GPU Memory: {max_memory} GB total, {start_gpu_memory} GB reserved before training")

# Train
trainer_stats = trainer.train()

# Check memory after training
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
print(f"\nTraining complete!")
print(f"Peak GPU memory used: {used_memory} GB / {max_memory} GB ({round(used_memory/max_memory*100, 1)}%)")
print(f"Training time: {trainer_stats.metrics['train_runtime']:.1f} seconds")
print(f"Final loss: {trainer_stats.metrics['train_loss']:.4f}")

Map:   0%|          | 0/51760 [00:00<?, ? examples/s]

GPU: Tesla T4
GPU Memory: 14.563 GB total, 3.07 GB reserved before training


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 51,760 | Num Epochs = 1 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 24,313,856 of 3,237,063,680 (0.75% trained)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 3


wandb: You chose "Don't visualize my results"
wandb: Using W&B in offline mode.
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


wandb: Detected [huggingface_hub.inference, openai] in use.
wandb: Use W&B Weave for improved LLM call tracing. Install Weave with `pip install weave` then add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai/


Step,Training Loss
1,1.701500
2,2.087700
3,1.811000
4,2.056800
5,1.837200
6,1.643700
7,1.256400
8,1.458500
9,1.325300
10,1.254800


train/epoch,▁▁▁▁▁▂▂▂▂▃▃▃▃▃▃▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▆▆▇▇▇▇▇███
train/global_step,▁▁▁▁▁▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▆▇▇▇▇▇███
train/grad_norm,▅█▅▆▅▄▆▇▅▃▂▂▃▃▅▅▄▇▄▄▆▄▅▁▁▁▂▂▂▁▁▂▂▂▁▂▁▂▁▁
train/learning_rate,▁▂▄▅▇██▇▇▇▇▇▇▆▆▆▆▅▅▅▅▅▄▄▄▄▄▄▃▃▃▃▃▃▂▂▂▂▂▁
train/loss,▆█▇█▆▅▄▄▂▃▂▂▅▃▂▃▃▃▃▃▂▂▂▃▂▂▂▁▃▄▂▂▃▁▄▃▂▄▂▁
total_flos,2157371736207360.0
train/epoch,0.00927
train/global_step,60
train/grad_norm,0.20082
train/learning_rate,0.0
train/loss,0.9675



Training complete!
Peak GPU memory used: 6.182 GB / 14.563 GB (42.5%)
Training time: 327.7 seconds
Final loss: 1.1423


In [8]:
# Switch to inference mode (2x faster)
FastLanguageModel.for_inference(model)

# Test with a prompt
test_prompt = alpaca_prompt.format(
    "Explain the concept of retrieval-augmented generation (RAG) to a product manager.",  # instruction
    "",  # input
    "",  # leave response blank - model will generate
)

inputs = tokenizer([test_prompt], return_tensors="pt").to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer)

print("=" * 50)
print("FINE-TUNED MODEL RESPONSE:")
print("=" * 50)
_ = model.generate(
    **inputs,
    streamer = text_streamer,
    max_new_tokens = 256,
    temperature = 0.7,
    top_p = 0.9,
)


FINE-TUNED MODEL RESPONSE:
<|begin_of_text|>Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
Explain the concept of retrieval-augmented generation (RAG) to a product manager.

### Input:


### Response:
Retrieval-Augmented Generation (RAG) is a method of generating text, particularly in the context of conversational AI and language models. It involves using a pre-trained language model to generate text based on the input provided by a retrieval system, which is essentially an AI-powered search engine that retrieves relevant information from a database.

The process of RAG works as follows:

1. The user inputs a prompt or a query to the model, which then uses its language understanding capabilities to understand the context and intent of the input.

2. The model then queries a retrieval system to retrieve relevant information related to the input, which may incl

In [9]:
# This is what you'd discuss in interviews about GPU optimization awareness
print("=" * 50)
print("GPU PERFORMANCE SUMMARY")
print("=" * 50)
print(f"GPU Model: {torch.cuda.get_device_properties(0).name}")
print(f"CUDA Version: {torch.version.cuda}")
print(f"Total GPU Memory: {max_memory} GB")
print(f"Peak Memory Used: {used_memory} GB ({round(used_memory/max_memory*100,1)}%)")
print(f"Training Time: {trainer_stats.metrics['train_runtime']:.1f}s")
print(f"Samples/Second: {trainer_stats.metrics['train_samples_per_second']:.2f}")
print(f"Steps/Second: {trainer_stats.metrics['train_steps_per_second']:.2f}")
print(f"Final Loss: {trainer_stats.metrics['train_loss']:.4f}")
print(f"Quantization: 4-bit NF4")
print(f"LoRA Rank: 16")
print(f"Trainable Params: ~16M / 3B ({round(16/3000*100, 2)}%)")
print(f"Effective Batch Size: {2 * 4} (batch_size={2} x grad_accum={4})")
print(f"Mixed Precision: {'bf16' if is_bfloat16_supported() else 'fp16'}")

GPU PERFORMANCE SUMMARY
GPU Model: Tesla T4
CUDA Version: 12.8
Total GPU Memory: 14.563 GB
Peak Memory Used: 6.182 GB (42.5%)
Training Time: 327.7s
Samples/Second: 1.47
Steps/Second: 0.18
Final Loss: 1.1423
Quantization: 4-bit NF4
LoRA Rank: 16
Trainable Params: ~16M / 3B (0.53%)
Effective Batch Size: 8 (batch_size=2 x grad_accum=4)
Mixed Precision: fp16


In [10]:
# Save LoRA adapters (small, ~33MB)
model.save_pretrained("lora_model")
tokenizer.save_pretrained("lora_model")

# Export to GGUF format (for local inference with llama.cpp)
# model.save_pretrained_gguf("model_gguf", tokenizer, quantization_method="q4_k_m")

print("Model saved! LoRA adapter size:")
!du -sh lora_model/

Model saved! LoRA adapter size:
110M	lora_model/
